In [3]:
!pip install ultralytics opencv-python -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.8 MB/s eta 0:00:00


In [ ]:
import torch

print("GPU disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No hay GPU activa")

GPU disponible: True
GPU: Tesla T4


In [4]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [5]:
from pathlib import Path
import os

BASE_DIR = Path("/content/drive/MyDrive/tesis_yolo")
ZIP_PATH = BASE_DIR / "Dataset.zip"
DATASET_ORIGINAL = BASE_DIR / "dataset_original"
YOLO_DATASET = BASE_DIR / "yolo_no_retractors_patient_split"
EXPERIMENTOS_DIR = BASE_DIR / "experimentos_patient_split"

BASE_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta base:", BASE_DIR)

Carpeta base: /content/drive/MyDrive/tesis_yolo


In [ ]:
!wget -O "/content/drive/MyDrive/tesis_yolo/Dataset.zip" "https://zenodo.org/records/14827784/files/Dataset.zip?download=1"

--2026-05-10 16:19:04--  https://zenodo.org/records/14827784/files/Dataset.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 137.138.153.219, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1576115314 (1.5G) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/tesis_yolo/Dataset.zip’

/content/drive/MyDr 100%[===================>]   1.47G  18.9MB/s    in 80s     

2026-05-10 16:20:25 (18.7 MB/s) - ‘/content/drive/MyDrive/tesis_yolo/Dataset.zip’ saved [1576115314/1576115314]



In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/tesis_yolo/Dataset.zip")
DATASET_ORIGINAL = Path("/content/drive/MyDrive/tesis_yolo/dataset_original")

if not DATASET_ORIGINAL.exists():
    DATASET_ORIGINAL.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(DATASET_ORIGINAL)

print("Dataset descomprimido en:", DATASET_ORIGINAL)

KeyboardInterrupt: 

In [6]:
import os, glob, shutil, random, re
from pathlib import Path
from collections import defaultdict

random.seed(42)

img_root = Path("/content/drive/MyDrive/tesis_yolo/dataset_original/Dataset/Images/no_retractors")
label_root = Path("/content/drive/MyDrive/tesis_yolo/dataset_original/Dataset/Annotations/Darknet_YOLO/no_retractors")

out_root = Path("/content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split")

if out_root.exists():
    shutil.rmtree(out_root)

for split in ["train", "val", "test"]:
    (out_root / "images" / split).mkdir(parents=True, exist_ok=True)
    (out_root / "labels" / split).mkdir(parents=True, exist_ok=True)

def get_patient_id(path):
    stem = Path(path).stem
    match = re.search(r"(anonymous[_-]\d+[_-]\d+[_-]\d+[_-]\d+)", stem)
    if match:
        return match.group(1).replace("_", "-")
    return stem

images = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    images.extend(img_root.glob(f"**/{ext}"))

patient_groups = defaultdict(list)

for img in images:
    pid = get_patient_id(img.name)
    patient_groups[pid].append(img)

patient_ids = list(patient_groups.keys())
random.shuffle(patient_ids)

n = len(patient_ids)
train_end = int(n * 0.7)
val_end = int(n * 0.9)

split_ids = {
    "train": set(patient_ids[:train_end]),
    "val": set(patient_ids[train_end:val_end]),
    "test": set(patient_ids[val_end:])
}

for split, ids in split_ids.items():
    for pid in ids:
        for img in patient_groups[pid]:
            rel = img.relative_to(img_root)
            label = label_root / rel.with_suffix(".txt")

            new_img_name = f"{rel.parent.name}_{img.name}"
            new_label_name = Path(new_img_name).with_suffix(".txt").name

            shutil.copy(img, out_root / "images" / split / new_img_name)

            if label.exists():
                shutil.copy(label, out_root / "labels" / split / new_label_name)
            else:
                open(out_root / "labels" / split / new_label_name, "w").close()

print("Dataset YOLO con split por paciente creado en:", out_root)

for split in ["train", "val", "test"]:
    imgs = list((out_root / "images" / split).glob("*"))
    labels = list((out_root / "labels" / split).glob("*"))
    ids = set(get_patient_id(p.name) for p in imgs)
    print(split, "imagenes:", len(imgs), "labels:", len(labels), "pacientes:", len(ids))

train_ids = set(get_patient_id(p.name) for p in (out_root / "images/train").glob("*"))
val_ids = set(get_patient_id(p.name) for p in (out_root / "images/val").glob("*"))
test_ids = set(get_patient_id(p.name) for p in (out_root / "images/test").glob("*"))

print("\nCruces de pacientes:")
print("train ∩ val:", len(train_ids & val_ids))
print("train ∩ test:", len(train_ids & test_ids))
print("val ∩ test:", len(val_ids & test_ids))

Dataset YOLO con split por paciente creado en: /content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split
train imagenes: 0 labels: 0 pacientes: 0
val imagenes: 0 labels: 0 pacientes: 0
test imagenes: 0 labels: 0 pacientes: 0

Cruces de pacientes:
train ∩ val: 0
train ∩ test: 0
val ∩ test: 0


In [7]:
yaml_content = """
path: /content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split
train: images/train
val: images/val
test: images/test

names:
  0: primary_caries
  1: permanent_caries
"""

with open("/content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split/data.yaml", "w") as f:
    f.write(yaml_content)

print(open("/content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split/data.yaml").read())


path: /content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split
train: images/train
val: images/val
test: images/test

names:
  0: primary_caries
  1: permanent_caries



In [ ]:
from ultralytics import YOLO

DATA_YAML = "/content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/tesis_yolo/experimentos_patient_split"

model = YOLO("yolov8s.pt")

results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=0.001,
    project=PROJECT_DIR,
    name="yolov8s_patient_split_adamw",
    exist_ok=True
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_patient_split_adamw, nbs=64, nms=False, opset=No

In [8]:
from ultralytics import YOLO
from pathlib import Path
import random
import matplotlib.pyplot as plt
import pandas as pd

MODEL_PATH = "/content/drive/MyDrive/tesis_yolo/experimentos_patient_split/yolov8s_patient_split_adamw/weights/best.pt"
TEST_DIR = Path("/content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split/images/test")

model = YOLO(MODEL_PATH)

test_images = (
    list(TEST_DIR.glob("*.jpg")) +
    list(TEST_DIR.glob("*.jpeg")) +
    list(TEST_DIR.glob("*.png")) +
    list(TEST_DIR.glob("*.JPG")) +
    list(TEST_DIR.glob("*.JPEG")) +
    list(TEST_DIR.glob("*.PNG"))
)

img_path = random.choice(test_images)

print("Imagen seleccionada del test:")
print(img_path)

results = model.predict(
    source=str(img_path),
    imgsz=640,
    conf=0.25,
    verbose=False
)

annotated_img = results[0].plot()

plt.figure(figsize=(10, 8))
plt.imshow(annotated_img)
plt.axis("off")
plt.title("Resultado preliminar de detección de posibles caries visibles")
plt.show()

detections = []

for box in results[0].boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    x1, y1, x2, y2 = box.xyxy[0].tolist()

    detections.append({
        "Resultado": "Posible caries visible",
        "Tipo": model.names[cls_id],
        "Confianza": round(conf, 3),
        "Ubicación": f"x1={round(x1)}, y1={round(y1)}, x2={round(x2)}, y2={round(y2)}"
    })

if detections:
    print("Zonas sospechosas detectadas:")
    display(pd.DataFrame(detections))
else:
    print("No se detectaron zonas sospechosas con el umbral actual.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/tesis_yolo/experimentos_patient_split/yolov8s_patient_split_adamw/weights/best.pt'

In [9]:
from ultralytics import YOLO
from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt

MODEL_PATH = "/content/drive/MyDrive/tesis_yolo/experimentos_patient_split/yolov8s_patient_split_adamw/weights/best.pt"

TEST_DIR = Path("/content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split/images/test")
LABEL_DIR = Path("/content/drive/MyDrive/tesis_yolo/yolo_no_retractors_patient_split/labels/test")

model = YOLO(MODEL_PATH)

test_images = (
    list(TEST_DIR.glob("*.jpg")) +
    list(TEST_DIR.glob("*.jpeg")) +
    list(TEST_DIR.glob("*.png")) +
    list(TEST_DIR.glob("*.JPG")) +
    list(TEST_DIR.glob("*.JPEG")) +
    list(TEST_DIR.glob("*.PNG"))
)

IMG_PATH = random.choice(test_images)
LABEL_PATH = LABEL_DIR / (IMG_PATH.stem + ".txt")

print("Imagen seleccionada:")
print(IMG_PATH)

img = cv2.imread(str(IMG_PATH))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w, _ = img.shape

real_boxes = []

if LABEL_PATH.exists():
    for line in LABEL_PATH.read_text().strip().splitlines():
        cls, x, y, bw, bh = map(float, line.split())

        x1 = int((x - bw / 2) * w)
        y1 = int((y - bh / 2) * h)
        x2 = int((x + bw / 2) * w)
        y2 = int((y + bh / 2) * h)

        real_boxes.append((int(cls), x1, y1, x2, y2))

        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 3)
        cv2.putText(img, f"REAL {int(cls)}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

results = model.predict(
    source=str(IMG_PATH),
    imgsz=640,
    conf=0.25,
    verbose=False
)

pred_boxes = []

for box in results[0].boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    x1, y1, x2, y2 = box.xyxy[0].tolist()

    pred_boxes.append((cls_id, conf, x1, y1, x2, y2))

    cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 3)
    cv2.putText(img, f"PRED {cls_id} {conf:.2f}", (int(x1), int(y2) + 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.title("Rojo = etiqueta real | Verde = predicción del modelo")
plt.show()

print("Cantidad de cajas reales:", len(real_boxes))
print("Cantidad de predicciones:", len(pred_boxes))
print("Label real:")
print(LABEL_PATH.read_text() if LABEL_PATH.exists() else "No existe label")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/tesis_yolo/experimentos_patient_split/yolov8s_patient_split_adamw/weights/best.pt'